# Week 06. FastAPI를 위한 Python 백엔드 기초

6주차는 데코레이터, 타입 힌트, Pydantic, SQLite CRUD처럼 FastAPI 전에 알아야 할 기초를 다룹니다.
이 노트북은 서버를 띄우지 않고도 핵심 개념이 실행되도록 구성했습니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. 데코레이터로 함수 등록하기

FastAPI의 `@app.get()`도 본질적으로는 함수를 특정 경로에 등록하는 데코레이터입니다.
아래 예제는 라우터 등록 구조를 작게 재현합니다.


In [1]:
routes = {}


def route(path):
    def decorator(func):
        routes[path] = func
        return func
    return decorator


@route("/hello")
def hello(name: str = "student") -> str:
    return f"Hello, {name}!"


print(routes["/hello"]("Sungmin"))
print("등록된 경로:", sorted(routes))


Hello, Sungmin!
등록된 경로: ['/hello']


## 2. Pydantic으로 입력 검증하기

웹 요청 데이터는 문자열, 누락값, 범위 오류가 섞여 들어올 수 있습니다.
Pydantic 모델은 이런 입력을 실행 시점에 검증합니다.


In [2]:
from pydantic import BaseModel, Field, ValidationError


class Customer(BaseModel):
    name: str = Field(min_length=1)
    age: int = Field(ge=1, le=120)
    grade: str = Field(pattern="^(basic|silver|gold)$")


valid_customer = Customer(name="Sungmin", age=21, grade="gold")
print(valid_customer)

try:
    Customer(name="", age=150, grade="vip")
except ValidationError as error:
    print("검증 오류 개수:", len(error.errors()))


name='Sungmin' age=21 grade='gold'
검증 오류 개수: 3


## 3. SQLite CRUD 흐름

CRUD는 Create, Read, Update, Delete의 약자입니다.
여기서는 메모리 DB를 써서 파일을 남기지 않고도 전체 흐름을 확인합니다.


In [3]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, age INTEGER, grade TEXT)")

cur.execute(
    "INSERT INTO customers (name, age, grade) VALUES (?, ?, ?)",
    (valid_customer.name, valid_customer.age, valid_customer.grade),
)
conn.commit()

cur.execute("UPDATE customers SET grade = ? WHERE name = ?", ("silver", "Sungmin"))
cur.execute("SELECT id, name, age, grade FROM customers")
rows = cur.fetchall()

cur.execute("DELETE FROM customers WHERE name = ?", ("Sungmin",))
conn.commit()

print("수정 후 조회:", rows)
print("삭제 후 건수:", cur.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
conn.close()


수정 후 조회: [(1, 'Sungmin', 21, 'silver')]
삭제 후 건수: 0


## 정리

- 데코레이터는 함수를 등록하거나 감싸는 패턴입니다.
- Pydantic은 요청 데이터 검증의 기반입니다.
- SQLite CRUD는 API가 실제 데이터를 다루는 기본 흐름입니다.
